# 01 — Type Hints & Annotations

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/01_type_hints_and_annotations.ipynb)

## 📌 What is it?
Type hints are optional annotations that describe the expected types of variables, function parameters, and return values. Python's `typing` module provides rich type constructs.

## 🧠 Why AI Engineers need this
Production AI codebases use type hints for IDE autocomplete, static analysis (mypy), and documentation. Libraries like Pydantic rely on them for data validation.

In [ ]:
# ── BASIC TYPE HINTS ──
from typing import Optional, Union, List, Dict, Tuple, Set, Any

# Variables
name: str = "Claude"
temperature: float = 0.7
max_tokens: int = 2048
is_streaming: bool = True
model: Optional[str] = None  # can be str or None

# Functions
def get_embedding(text: str, dim: int = 1536) -> list[float]:
    """Returns an embedding vector."""
    return [0.0] * dim

def parse_response(data: dict[str, Any]) -> Optional[str]:
    """Extract text from API response, returns None if missing."""
    return data.get("content", [{}])[0].get("text")

result = get_embedding("Hello")
print(f"Embedding dims: {len(result)}")

In [ ]:
# ── UNION and OPTIONAL ──
from typing import Union, Optional

# Union — multiple allowed types
def set_temperature(value: Union[int, float]) -> float:
    return float(value)

# Optional[X] is shorthand for Union[X, None]
def load_config(path: Optional[str] = None) -> dict:
    if path is None:
        return {"model": "gpt-4o", "temperature": 0.7}
    import json
    with open(path) as f:
        return json.load(f)

# Python 3.10+ shorthand:
def new_style(value: int | float | None) -> float | None:
    return float(value) if value is not None else None

In [ ]:
# ── CALLABLE, TYPEVAR, GENERICS ──
from typing import Callable, TypeVar, Generic, Sequence

# Callable[[arg_types], return_type]
TransformFn = Callable[[str], str]

def apply_transforms(text: str, transforms: list[TransformFn]) -> str:
    for fn in transforms:
        text = fn(text)
    return text

def lower(s: str) -> str: return s.lower()
def strip(s: str) -> str: return s.strip()

result = apply_transforms("  Hello WORLD  ", [strip, lower])
print(result)  # hello world

# TypeVar — generic functions
T = TypeVar('T')

def first(items: Sequence[T]) -> Optional[T]:
    return items[0] if items else None

print(first([1, 2, 3]))        # 1
print(first(["a", "b", "c"])) # a
print(first([]))               # None

In [ ]:
# ── PYDANTIC — runtime type validation ──
# pip install pydantic

from pydantic import BaseModel, Field, field_validator
from typing import Optional, Literal

class LLMRequest(BaseModel):
    prompt: str
    model: Literal["gpt-4o", "claude-3-5-sonnet", "gemini-pro"] = "gpt-4o"
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1000, gt=0, le=128000)
    system_prompt: Optional[str] = None
    
    @field_validator("prompt")
    def prompt_not_empty(cls, v):
        if not v.strip():
            raise ValueError("Prompt cannot be empty")
        return v.strip()

# Valid
req = LLMRequest(prompt="What is Python?", temperature=0.5)
print(req.model_dump())

# Invalid — will raise ValidationError
try:
    bad = LLMRequest(prompt="   ", temperature=5.0)
except Exception as e:
    print(f"Validation error: {e}")

## ⚠️ Common Mistakes
```python
# ❌ Type hints are NOT enforced at runtime by default
def add(a: int, b: int) -> int:
    return a + b

add("hello", " world")  # works! (returns 'hello world')
# Use mypy or pydantic for actual enforcement

# ❌ list vs List (Python 3.9+)
from typing import List  # old style
def f(x: List[int]): ...  # still works
def f(x: list[int]): ...  # ✅ modern (3.9+)
```

## 🔗 What's Next?
[02 — Dataclasses →](02_dataclasses.ipynb)